# Chapter 29: Tracking, Logging & Checkpointing

        **Part V - The Training Loop, Mastered** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Record meaningful training history
- Save the best model state
- Resume from a full checkpoint


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## History is structured data

Store one comparable scalar per epoch so curves, CSV files, and experiment summaries come from the same source.


In [ ]:
history = {"train_loss": [], "val_loss": []}
for epoch in range(8):
    history["train_loss"].append(1.0 / (epoch + 1))
    history["val_loss"].append(0.2 + 0.9 / (epoch + 1))
print(history)


## Save and restore weights

Save state_dict rather than the whole model object. Loading requires recreating the architecture.


In [ ]:
import tempfile
from pathlib import Path
import torch.nn as nn

workdir = Path(tempfile.mkdtemp(prefix="pytorch-book-"))
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
sample = torch.randn(3, 4)
before = model(sample).detach()
weights_path = workdir / "model.pt"
torch.save(model.state_dict(), weights_path)
restored = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
restored.load_state_dict(torch.load(weights_path, weights_only=True))
print("same output:", torch.allclose(before, restored(sample)))


## A resumable checkpoint

A full training checkpoint combines model, optimizer, epoch, best score, and history. These are trusted local objects, so weights_only=False is intentional here.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
checkpoint_path = workdir / "checkpoint.pt"
torch.save({
    "epoch": 7,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "best_val": min(history["val_loss"]),
    "history": history,
}, checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)
model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])
print("resume at epoch:", checkpoint["epoch"] + 1, "best:", checkpoint["best_val"])


## Practice

        Work through these without looking back first.

        1. Write history to a CSV file.
2. Save only when validation loss improves.
3. Add a scheduler state to the checkpoint.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 29's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
